# SegFormer-B2 Maritime Segmentation (Colab)

Train on **LaRS + MaSTr1325** from Google Drive.  
Data: `MaritimeSegmentation/datasets/` (lars, mastr1325).  
Checkpoints: `MaritimeSegmentation/checkpoints/`.

Set runtime to **GPU** (T4). Run cells in order.

In [ ]:
# Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Base folder on Drive: MaritimeSegmentation
DRIVE_BASE = Path("/content/drive/MyDrive/MaritimeSegmentation")
DATASETS_ROOT = DRIVE_BASE / "datasets"
CHECKPOINTS_DIR = DRIVE_BASE / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

LARS_IMAGES = DATASETS_ROOT / "lars" / "lars_images"
LARS_ANNOTATIONS = DATASETS_ROOT / "lars" / "lars_annotations"
MASTR_IMAGES = DATASETS_ROOT / "mastr1325" / "MaSTr1325_images_512x384"
MASTR_MASKS = DATASETS_ROOT / "mastr1325" / "MaSTr1325_masks_512x384"

print("LARS exists:", LARS_IMAGES.exists())
print("MaSTr exists:", MASTR_IMAGES.exists())
print("Checkpoints dir:", CHECKPOINTS_DIR)

In [ ]:
!pip install -q transformers timm albumentations

In [ ]:
# Config — 3-class unified schema (Sky, Water, Obstacle)
# WHY 3 CLASSES: MaSTr's class 0 is "Obstacles & Environment" (mapped to Land before),
# while LaRS's class 0 is "Obstacles" (mapped to Obstacle). This caused the same visual
# objects (ships, cranes, buoys) to receive contradictory labels depending on the dataset,
# making the model learn dataset style instead of semantics. With 3 classes, both datasets
# map identically: {0->Obstacle, 1->Water, 2->Sky} — fully consistent supervision.
NUM_CLASSES = 3
CLASS_NAMES = ("Sky", "Water", "Obstacle")
IGNORE_INDEX = 255
LARS_TO_UNIFIED  = {0: 2, 1: 1, 2: 0}   # Obstacles->2, Water->1, Sky->0
MASTR_TO_UNIFIED = {0: 2, 1: 1, 2: 0}   # Env/Obstacles->2, Water->1, Sky->0  (same as LaRS)
MASTR_IGNORE_VALUE = 4

# Training
BATCH_SIZE = 8
EPOCHS = 60
LR = 6e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0          # max gradient norm; 0 to disable
# Class weights for CrossEntropyLoss — boost obstacle (sparse in open-ocean shots).
# Set to None for unweighted. Tune based on pixel-frequency inspection.
CLASS_WEIGHTS = [1.0, 1.0, 2.0]  # Sky, Water, Obstacle

# Early stopping: stop if val mIoU hasn't improved for this many epochs.
# In the last run best was epoch 29 → with patience=10 we'd have stopped at epoch 39.
EARLY_STOP_PATIENCE = 10

INPUT_HEIGHT, INPUT_WIDTH = 384, 512
NUM_WORKERS = 2
SAVE_EVERY_N_EPOCHS = 5
VAL_EVERY_N_EPOCHS = 1


In [ ]:
# Dataset classes
import json
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, ConcatDataset

def _map_mastr_mask(mask):
    out = np.full_like(mask, IGNORE_INDEX, dtype=np.int64)
    for src, dst in MASTR_TO_UNIFIED.items():
        out[mask == src] = dst
    out[mask == MASTR_IGNORE_VALUE] = IGNORE_INDEX
    return out

def _map_lars_mask(mask):
    out = np.full_like(mask, IGNORE_INDEX, dtype=np.int64)
    for src, dst in LARS_TO_UNIFIED.items():
        out[mask == src] = dst
    out[mask == 255] = IGNORE_INDEX
    return out

class MaSTr1325Dataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform
        self.samples = []
        for p in sorted(self.images_dir.glob("*.jpg")):
            mask_path = self.masks_dir / f"{p.stem}m.png"
            if mask_path.exists():
                self.samples.append((p, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = _map_mastr_mask(np.array(Image.open(mask_path)))
        if self.transform:
            out = self.transform(image=image, mask=mask)
            image, mask = out["image"], out["mask"]
        mask = torch.from_numpy(mask).long() if isinstance(mask, np.ndarray) else mask.long()
        return {"image": image, "mask": mask}

class LaRSDataset(Dataset):
    def __init__(self, split, images_root, annotations_root, transform=None):
        self.images_dir = Path(images_root) / split / "images"
        self.masks_dir = Path(annotations_root) / split / "semantic_masks"
        self.transform = transform
        with open(Path(annotations_root) / split / "image_annotations.json") as f:
            data = json.load(f)
        self.samples = []
        for a in data.get("annotations", []):
            fn = a["file_name"]
            img_path = self.images_dir / fn
            mask_path = self.masks_dir / (Path(fn).stem + ".png")
            if img_path.exists() and mask_path.exists():
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = _map_lars_mask(np.array(Image.open(mask_path)))
        if self.transform:
            out = self.transform(image=image, mask=mask)
            image, mask = out["image"], out["mask"]
        mask = torch.from_numpy(mask).long() if isinstance(mask, np.ndarray) else mask.long()
        return {"image": image, "mask": mask}

def CombinedMaritimeDataset(split, transform=None, use_mastr=True, use_lars=True):
    datasets = []
    if use_mastr and MASTR_IMAGES.exists():
        full = MaSTr1325Dataset(MASTR_IMAGES, MASTR_MASKS, transform)
        n = len(full)
        if n > 0:
            val_size = max(1, n // 10)
            idx = range(0, n - val_size) if split == "train" else range(n - val_size, n)
            datasets.append(torch.utils.data.Subset(full, idx))
    if use_lars and LARS_IMAGES.exists():
        ds = LaRSDataset(split, LARS_IMAGES, LARS_ANNOTATIONS, transform)
        if len(ds) > 0:
            datasets.append(ds)
    if not datasets:
        raise FileNotFoundError("No data. Check Drive path: MaritimeSegmentation/datasets/lars and mastr1325")
    return ConcatDataset(datasets)

In [ ]:
# Transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.Resize(INPUT_HEIGHT, INPUT_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.OneOf([A.MotionBlur(p=0.3), A.GaussianBlur(blur_limit=3, p=0.3)], p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])
val_tf = A.Compose([
    A.Resize(INPUT_HEIGHT, INPUT_WIDTH),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])

train_ds = CombinedMaritimeDataset("train", transform=train_tf)
val_ds = CombinedMaritimeDataset("val", transform=val_tf)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Model — 3-class decoder
from transformers import SegformerForSemanticSegmentation
import torch

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Device: {device}, NUM_CLASSES: {NUM_CLASSES}")


In [ ]:
# Training loop  (AMP + gradient clipping + early stopping)
import torch.nn as nn
import torch.nn.functional as F

weights = torch.tensor(CLASS_WEIGHTS, device=device) if CLASS_WEIGHTS else None
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX, weight=weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: (1 - e / EPOCHS) ** 0.9)

# AMP scaler — uses fp16 on GPU, no-op on CPU
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

best_miou = 0.0
no_improve_epochs = 0
best_ckpt_path = CHECKPOINTS_DIR / "best_segformer_b2_maritime.pt"

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        masks  = batch["mask"].to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            out    = model(pixel_values=images)
            logits = F.interpolate(out.logits, size=masks.shape[1:], mode="bilinear", align_corners=False)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    if (epoch + 1) % VAL_EVERY_N_EPOCHS == 0:
        model.eval()
        val_loss = 0.0
        correct = total_pixels = 0
        class_correct = [0] * NUM_CLASSES
        class_total   = [0] * NUM_CLASSES
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                masks  = batch["mask"].to(device)
                with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                    out    = model(pixel_values=images)
                    logits = F.interpolate(out.logits, size=masks.shape[1:], mode="bilinear", align_corners=False)
                val_loss += criterion(logits, masks).item()
                pred  = logits.argmax(dim=1)
                valid = masks != IGNORE_INDEX
                correct      += (pred[valid] == masks[valid]).sum().item()
                total_pixels += valid.sum().item()
                for c in range(NUM_CLASSES):
                    m = masks == c
                    if m.any():
                        class_total[c]   += m.sum().item()
                        class_correct[c] += (pred[m] == c).sum().item()

        val_loss /= max(len(val_loader), 1)
        acc  = correct / max(total_pixels, 1)
        ious = [class_correct[c] / class_total[c] for c in range(NUM_CLASSES) if class_total[c] > 0]
        miou = sum(ious) / max(len(ious), 1) if ious else 0.0
        per_cls = "  ".join(
            f"{CLASS_NAMES[c]}={class_correct[c]/class_total[c]:.3f}" if class_total[c] > 0 else f"{CLASS_NAMES[c]}=N/A"
            for c in range(NUM_CLASSES)
        )
        print(f"Epoch {epoch+1}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  acc={acc:.4f}  mIoU={miou:.4f}  [{per_cls}]")

        # ── Checkpoint & early stopping ────────────────────────────────────
        if miou > best_miou:
            best_miou = miou
            no_improve_epochs = 0
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "miou": miou}, best_ckpt_path)
            print(f"  -> new best mIoU={best_miou:.4f}, saved to {best_ckpt_path}")
        else:
            no_improve_epochs += VAL_EVERY_N_EPOCHS
            print(f"  no improvement ({no_improve_epochs}/{EARLY_STOP_PATIENCE} patience)")
            if no_improve_epochs >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch+1}. Best mIoU: {best_miou:.4f}")
                break

    # ── Periodic checkpoint ────────────────────────────────────────────────
    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict()},
                   CHECKPOINTS_DIR / f"segformer_b2_epoch_{epoch+1}.pt")

print(f"\nTraining done. Best mIoU: {best_miou:.4f}  |  checkpoint: {best_ckpt_path}")


## Evaluation on LaRS test set

Runs **after training** using the best saved checkpoint.  
LaRS test split was never seen during training or validation — it is a clean hold-out.

- If `lars_annotations/test/semantic_masks/` exists → full per-class IoU / mIoU / pixel accuracy.  
- If masks are absent → inference-only with predicted class distribution (useful to spot class collapse).

Results and overlay images are saved to `MaritimeSegmentation/test_results/`.


In [ ]:
# ── Test on LaRS (clean hold-out set) ──────────────────────────────────────
import json
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader

TEST_RESULTS_DIR = DRIVE_BASE / "test_results"
TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

CLASS_COLORS_BGR = [
    (180, 120, 255),   # Sky      (0) - purple
    (255, 200, 100),   # Water    (1) - teal
    (100, 100, 255),   # Obstacle (2) - red
]

# ── Load best checkpoint ───────────────────────────────────────────────────
ckpt = torch.load(best_ckpt_path, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']}, mIoU={ckpt.get('miou', '?'):.4f}")

# ── Build LaRS test dataset ─────────────────────────────────────────────────
test_masks_dir = LARS_ANNOTATIONS / "test" / "semantic_masks"
has_masks = test_masks_dir.exists() and any(test_masks_dir.glob("*.png"))

if has_masks:
    test_ds = LaRSDataset("test", LARS_IMAGES, LARS_ANNOTATIONS, transform=val_tf)
    print(f"LaRS test: {len(test_ds)} samples with ground-truth masks")
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    # Metric accumulators
    per_class_inter = [0] * NUM_CLASSES
    per_class_union = [0] * NUM_CLASSES
    total_correct = total_valid = 0

    for i, batch in enumerate(test_loader):
        images = batch["image"].to(device)
        masks  = batch["mask"].numpy()
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                out    = model(pixel_values=images)
                logits = F.interpolate(out.logits, size=masks.shape[1:], mode="bilinear", align_corners=False)
        preds = logits.argmax(dim=1).cpu().numpy()

        for b in range(images.size(0)):
            pred, mask = preds[b], masks[b]
            valid = mask != IGNORE_INDEX
            total_correct += (pred[valid] == mask[valid]).sum()
            total_valid   += valid.sum()
            for c in range(NUM_CLASSES):
                pc = pred == c
                mc = mask == c
                per_class_inter[c] += int(np.logical_and(pc, mc).sum())
                per_class_union[c] += int(np.logical_or(pc, mc).sum())

            # Save overlay
            img_t  = images[b].cpu()
            mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_np = (img_t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
            img_bgr = np.ascontiguousarray((img_np * 255).astype(np.uint8)[:, :, ::-1])
            overlay = img_bgr.copy()
            for c in range(NUM_CLASSES):
                overlay[pred == c] = CLASS_COLORS_BGR[c]
            overlay = (0.55 * img_bgr + 0.45 * overlay).astype(np.uint8)
            name = batch["name"][b] if "name" in batch else f"{i*BATCH_SIZE+b:05d}"
            cv2.imwrite(str(TEST_RESULTS_DIR / f"{name}_seg.png"), overlay)

    # Report
    pixel_acc = total_correct / max(total_valid, 1)
    class_ious = [per_class_inter[c] / per_class_union[c] if per_class_union[c] > 0 else float("nan")
                  for c in range(NUM_CLASSES)]
    valid_ious = [v for v in class_ious if not np.isnan(v)]
    miou_test  = sum(valid_ious) / len(valid_ious) if valid_ious else 0.0

    print(f"\nLaRS test results")
    print(f"  Pixel accuracy : {pixel_acc:.4f}")
    print(f"  mIoU           : {miou_test:.4f}")
    print( "  Per-class IoU  :")
    for c, name in enumerate(CLASS_NAMES):
        v = class_ious[c]
        print(f"    {name:10s}: {v:.4f}" if not np.isnan(v) else f"    {name:10s}: (no pixels in test set)")
    print(f"\nOverlays saved to {TEST_RESULTS_DIR}")

else:
    # No ground-truth masks — inference only
    ann_file = LARS_ANNOTATIONS / "test" / "image_annotations.json"
    with open(ann_file) as f:
        ann_data = json.load(f)
    test_img_paths = [
        LARS_IMAGES / "test" / "images" / a["file_name"]
        for a in ann_data.get("annotations", [])
        if (LARS_IMAGES / "test" / "images" / a["file_name"]).exists()
    ]
    print(f"No test masks found. Running inference on {len(test_img_paths)} images.")

    class_pixels = [0] * NUM_CLASSES
    total_pixels = 0

    for img_path in test_img_paths:
        img = cv2.imread(str(img_path))
        h_orig, w_orig = img.shape[:2]
        img_resized = cv2.resize(img, (INPUT_WIDTH, INPUT_HEIGHT)).astype(np.float32) / 255.0
        img_norm = (img_resized - IMAGENET_MEAN) / IMAGENET_STD
        x = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device == "cuda")):
                out    = model(pixel_values=x)
                logits = F.interpolate(out.logits, size=(h_orig, w_orig), mode="bilinear", align_corners=False)
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy()
        for c in range(NUM_CLASSES):
            class_pixels[c] += int((pred == c).sum())
        total_pixels += pred.size

        # Save overlay
        overlay = img.copy()
        for c in range(NUM_CLASSES):
            overlay[pred == c] = CLASS_COLORS_BGR[c]
        result = (0.55 * img + 0.45 * overlay).astype(np.uint8)
        cv2.imwrite(str(TEST_RESULTS_DIR / (img_path.stem + "_seg.png")), result)

    print(f"\nPredicted class distribution (inference-only):")
    for c, name in enumerate(CLASS_NAMES):
        pct = 100.0 * class_pixels[c] / max(total_pixels, 1)
        print(f"  {name:10s} ({c}): {pct:.1f}%")
    print(f"\nOverlays saved to {TEST_RESULTS_DIR}")
